In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.10.2', 'jax==0.10.2', 'jax[cuda12]==0.10.2', 'exact'), 'jaxlib': ('0.10.2', 'jaxlib==0.10.2', 'jaxlib==0.10.2', 'exact'), 'flax': ('0.12.7', 'flax==0.12.7', 'flax==0.12.7', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8cd319b4f2187b6b29bb69603a96460fc325a975"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8cd319b4f2187b6b29bb69603a96460fc325a975/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Sentiment Analysis: Using Convolutional Neural Networks


In that section,
we investigated mechanisms
for processing
two-dimensional image data
with two-dimensional CNNs,
which were applied to
local features such as adjacent pixels.
Though originally
designed for computer vision,
CNNs are also widely used
for natural language processing.
Simply put,
just think of any text sequence
as a one-dimensional image.
In this way,
one-dimensional CNNs
can process local features
such as $n$-grams in text.

In this section,
we will use the *textCNN* model
to demonstrate
how to design a CNN architecture
for representing single text [@Kim.2014].
Compared with
the figure
that uses an RNN architecture with GloVe pretraining
for sentiment analysis,
the only difference in the figure
lies in
the choice of the architecture.


![This section feeds pretrained GloVe to a CNN-based architecture for sentiment analysis.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/nlp-map-sa-cnn.svg)

In [1]:
from d2l import jax as d2l
import jax
from jax import numpy as jnp
from flax import nnx
import optax
import numpy as np

batch_size = 256
train_iter, test_iter, vocab = d2l.load_data_imdb(batch_size)

## One-Dimensional Convolutions

Before introducing the model,
let's see how a one-dimensional convolution works.
Bear in mind that it is just a special case
of a two-dimensional convolution
based on the cross-correlation operation.

![One-dimensional cross-correlation operation. The shaded portions are the first output element as well as the input and kernel tensor elements used for the output computation: $0\times1+1\times2=2$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv1d.svg)

As shown in the figure,
in the one-dimensional case,
the convolution window
slides from left to right
across the input tensor.
During sliding,
the input subtensor (e.g., $0$ and $1$ in the figure) contained in the convolution window
at a certain position
and the kernel tensor (e.g., $1$ and $2$ in the figure) are multiplied elementwise.
The sum of these multiplications
gives the single scalar value (e.g., $0\times1+1\times2=2$ in the figure)
at the corresponding position of the output tensor.

We implement one-dimensional cross-correlation in the following `corr1d` function.
Given an input tensor `X`
and a kernel tensor `K`,
it returns the output tensor `Y`.

In [2]:
def corr1d(X, K):
    w = K.shape[0]
    Y = d2l.zeros((X.shape[0] - w + 1))
    for i in range(Y.shape[0]):
        Y = Y.at[i].set((X[i: i + w] * K).sum())
    return Y

We can construct the input tensor `X` and the kernel tensor `K` from the figure to validate the output of the above one-dimensional cross-correlation implementation.

In [3]:
X, K = d2l.tensor([0, 1, 2, 3, 4, 5, 6]), d2l.tensor([1, 2])
corr1d(X, K)

Array([ 2.,  5.,  8., 11., 14., 17.], dtype=float32)

For any
one-dimensional input with multiple channels,
the convolution kernel
needs to have the same number of input channels.
Then for each channel,
perform a cross-correlation operation on the one-dimensional tensor of the input and the one-dimensional tensor of the convolution kernel,
summing the results over all the channels
to produce the one-dimensional output tensor.
the figure shows a one-dimensional cross-correlation operation with 3 input channels.

![One-dimensional cross-correlation operation with 3 input channels. The shaded portions are the first output element as well as the input and kernel tensor elements used for the output computation: $0\times1+1\times2+1\times3+2\times4+2\times(-1)+3\times(-3)=2$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv1d-channel.svg)


We can implement the one-dimensional cross-correlation operation for multiple input channels
and validate the results in the figure.

In [4]:
def corr1d_multi_in(X, K):
    # First, iterate through the 0th dimension (channel dimension) of `X` and
    # `K`. Then, add them together
    return sum(corr1d(x, k) for x, k in zip(X, K))

X = d2l.tensor([[0, 1, 2, 3, 4, 5, 6],
              [1, 2, 3, 4, 5, 6, 7],
              [2, 3, 4, 5, 6, 7, 8]])
K = d2l.tensor([[1, 2], [3, 4], [-1, -3]])
corr1d_multi_in(X, K)

Array([ 2.,  8., 14., 20., 26., 32.], dtype=float32)

Note that
multi-input-channel one-dimensional cross-correlations
are equivalent
to
single-input-channel
two-dimensional cross-correlations.
To illustrate,
an equivalent form of
the multi-input-channel one-dimensional cross-correlation
in the figure
is
the
single-input-channel
two-dimensional cross-correlation
in the figure,
where the height of the convolution kernel
has to be the same as that of the input tensor.


![Two-dimensional cross-correlation operation with a single input channel. The shaded portions are the first output element as well as the input and kernel tensor elements used for the output computation: $2\times(-1)+3\times(-3)+1\times3+2\times4+0\times1+1\times2=2$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv1d-2d.svg)

Both the outputs in the figure and the figure have only one channel.
Same as two-dimensional convolutions with multiple output channels described in that section,
we can also specify multiple output channels
for one-dimensional convolutions.

## Max-Over-Time Pooling

Similarly, we can use pooling
to extract the highest value
from sequence representations
as the most important feature
across time steps.
The *max-over-time pooling* used in textCNN
works like
the one-dimensional global max-pooling
[@Collobert.Weston.Bottou.ea.2011].
For a multi-channel input
where each channel stores values
at different time steps,
the output at each channel
is the maximum value
for that channel.
Note that
the max-over-time pooling
allows different numbers of time steps
at different channels.

## The textCNN Model

Using the one-dimensional convolution
and max-over-time pooling,
the textCNN model
takes individual pretrained token representations
as input,
then obtains and transforms sequence representations
for the downstream application.

For a single text sequence
with $n$ tokens represented by
$d$-dimensional vectors,
the width, height, and number of channels
of the input tensor
are $n$, $1$, and $d$, respectively.
The textCNN model transforms the input
into the output as follows:

1. Define multiple one-dimensional convolution kernels and perform convolution operations separately on the inputs. Convolution kernels with different widths may capture local features among different numbers of adjacent tokens.
1. Perform max-over-time pooling on all the output channels, and then concatenate all the scalar pooling outputs as a vector.
1. Transform the concatenated vector into the output categories using the fully connected layer. Dropout can be used for reducing overfitting.

![The model architecture of textCNN.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/textcnn.svg)

the figure
illustrates the model architecture of textCNN
with a concrete example.
The input is a sentence with 11 tokens,
where
each token is represented by a 6-dimensional vector.
So we have a 6-channel input with width 11.
Define
two one-dimensional convolution kernels
of widths 2 and 4,
with 4 and 5 output channels, respectively.
They produce
4 output channels with width $11-2+1=10$
and 5 output channels with width $11-4+1=8$.
Despite different widths of these 9 channels,
the max-over-time pooling
gives a concatenated 9-dimensional vector,
which is finally transformed
into a 2-dimensional output vector
for binary sentiment predictions.


### Defining the Model

We implement the textCNN model in the following class.
Compared with the bidirectional RNN model in
that section,
besides
replacing recurrent layers with convolutional layers,
we also use two embedding layers:
one with trainable weights and the other
with fixed weights.

In [5]:
class TextCNN(nnx.Module):
    def __init__(self, vocab_size, embed_size, kernel_sizes, num_channels,
                 rngs=None):
        rngs = nnx.Rngs(params=0, dropout=1) if rngs is None else rngs
        self.embedding = nnx.Embed(vocab_size, embed_size, rngs=rngs)
        # The embedding layer not to be trained
        self.constant_embedding = nnx.Embed(vocab_size, embed_size, rngs=rngs)
        self.dropout = nnx.Dropout(0.5, rngs=rngs)
        self.decoder = nnx.Linear(sum(num_channels), 2, rngs=rngs)
        # Create multiple one-dimensional convolutional layers
        self.convs = nnx.List([
            nnx.Conv(2 * embed_size, c, kernel_size=(k,), rngs=rngs)
            for c, k in zip(num_channels, kernel_sizes)])

    def __call__(self, inputs):
        # Concatenate two embedding layer outputs with shape (batch size, no.
        # of tokens, token vector dimension) along vectors
        embeddings = jnp.concatenate((
            self.embedding(inputs), self.constant_embedding(inputs)), axis=2)
        # For Flax Conv, input shape is (batch, length, channels) which is
        # already the shape of embeddings
        # For each one-dimensional convolutional layer, after max-over-time
        # pooling, a tensor of shape (batch size, no. of channels) is obtained.
        # Concatenate along channels
        encoding = jnp.concatenate([
            jnp.max(nnx.relu(conv(embeddings)), axis=1)
            for conv in self.convs], axis=1)
        outputs = self.decoder(self.dropout(encoding))
        return outputs

Let's create a textCNN instance.
It has 3 convolutional layers with kernel widths of 3, 4, and 5, all with 100 output channels.

In [6]:
embed_size, kernel_sizes, nums_channels = 100, [3, 4, 5], [100, 100, 100]
devices = d2l.try_all_gpus()
net = TextCNN(len(vocab), embed_size, kernel_sizes, nums_channels)
d2l.check_shape(nnx.view(net, deterministic=True)(
    jnp.ones((1, 500), dtype=jnp.int32)), (1, 2))

### Loading Pretrained Word Vectors

Same as that section,
we load pretrained 100-dimensional GloVe embeddings
as the initialized token representations.
These token representations (embedding weights)
will be trained in `embedding`
and fixed in `constant_embedding`.

In [7]:
glove_embedding = d2l.TokenEmbedding('glove.6b.100d')
embeds = glove_embedding[vocab.idx_to_token]
# Set pretrained embedding weights in the parameters
net.embedding.embedding[...] = jnp.array(embeds)
net.constant_embedding.embedding = nnx.data(jnp.array(embeds))

### Training and Evaluating the Model

Now we can train the textCNN model for sentiment analysis.

In [8]:
lr, num_epochs = 0.001, 5
optimizer = nnx.Optimizer(net, optax.adam(lr), wrt=nnx.Param)
loss_fn = optax.softmax_cross_entropy_with_integer_labels

@nnx.jit
def train_step(net, optimizer, X, y):
    def compute_loss(model):
        logits = model(X)
        return loss_fn(logits, y).mean(), logits
    (loss, logits), grads = nnx.value_and_grad(
        compute_loss, has_aux=True)(net)
    optimizer.update(net, grads)
    return loss, logits

for epoch in range(num_epochs):
    loss_sum, train_correct, num_train = (
        jnp.array(0.0), jnp.array(0), 0)
    for X, y in train_iter:
        l, logits = train_step(net, optimizer, X, y)
        loss_sum += l * len(y)
        train_correct += (logits.argmax(axis=-1) == y).sum()
        num_train += len(y)
    # Evaluate
    correct, total = jnp.array(0), 0
    for X, y in test_iter:
        logits = nnx.view(net, deterministic=True)(X)
        correct += (logits.argmax(axis=-1) == y).sum()
        total += len(y)
    loss_sum, train_correct, correct = (
        float(loss_sum), int(train_correct), int(correct))
    print(f'epoch {epoch + 1}, loss {loss_sum / num_train:.3f}, '
          f'train acc {train_correct / num_train:.3f}, '
          f'test acc {correct / total:.3f}')

epoch 1, loss 0.644, train acc 0.664, test acc 0.812


epoch 2, loss 0.413, train acc 0.811, test acc 0.842


epoch 3, loss 0.339, train acc 0.851, test acc 0.856


epoch 4, loss 0.280, train acc 0.886, test acc 0.866


epoch 5, loss 0.219, train acc 0.916, test acc 0.875


Below we use the trained model to predict the sentiment for two simple sentences.

In [9]:
tokens = jnp.array(vocab['this movie is so great'.split()])
logits = nnx.view(net, deterministic=True)(tokens.reshape(1, -1))
'positive' if int(jnp.argmax(logits, axis=1)[0]) == 1 else 'negative'

'positive'

In [10]:
tokens = jnp.array(vocab['this movie is so bad'.split()])
logits = nnx.view(net, deterministic=True)(tokens.reshape(1, -1))
'positive' if int(jnp.argmax(logits, axis=1)[0]) == 1 else 'negative'

'negative'

## Summary

* One-dimensional CNNs can process local features such as $n$-grams in text.
* Multi-input-channel one-dimensional cross-correlations are equivalent to single-input-channel two-dimensional cross-correlations.
* The max-over-time pooling allows different numbers of time steps at different channels.
* The textCNN model transforms individual token representations into downstream application outputs using one-dimensional convolutional layers and max-over-time pooling layers.


## Exercises

1. Tune hyperparameters and compare the two architectures for sentiment analysis in that section and in this section, such as in classification accuracy and computational efficiency.
1. Can you further improve the classification accuracy of the model by using the methods introduced in the exercises of that section?
1. Add positional encoding in the input representations. Does it improve the classification accuracy?

[Discussions](https://d2l.discourse.group/t/1425)